In [1]:
import time
import numpy as np

from src.config import SimulationConfig
from src.field import ElectricField
from src.growth import get_front, get_growth_probabilities


In [2]:
config = SimulationConfig(
    nx=96,
    ny=96,
    voltage=1.0,
    tolerance=1e-5,
    max_iterations=50_000,
    omega=1.9,
)

def make_field():
    field = ElectricField(config)
    field.channel[:30, config.nx // 2] = True
    field.apply_boundary_conditions()
    return field

field_jacobi = make_field()
field_sor = make_field()

assert np.allclose(field_jacobi.phi, field_sor.phi)


In [3]:
start = time.perf_counter()
jacobi_iterations, jacobi_residual = field_jacobi.solve_laplace_jacobi_residual()
jacobi_time = time.perf_counter() - start

start = time.perf_counter()
sor_iterations, sor_residual = field_sor.solve_laplace_sor()
sor_time = time.perf_counter() - start


In [4]:
phi_difference = np.max(np.abs(field_jacobi.phi - field_sor.phi))

_, _, E_jacobi = field_jacobi.electric_field()
_, _, E_sor = field_sor.electric_field()
E_difference = np.max(np.abs(E_jacobi - E_sor))
front = get_front(field_jacobi.channel)
P_jacobi = get_growth_probabilities(E_jacobi, front, config.eta)
P_sor = get_growth_probabilities(E_sor, front, config.eta)
tv_distance = 0.5 * np.sum(np.abs(P_jacobi - P_sor))

speedup = jacobi_time / sor_time


In [5]:
print(f"{'solver':<16} {'iterations':>10} {'time, s':>12} {'residual':>14}")
print(f"{'Jacobi':<16} {jacobi_iterations:>10} {jacobi_time:>12.6f} {jacobi_residual:>14.3e}")
print(f"{'Red-Black SOR':<16} {sor_iterations:>10} {sor_time:>12.6f} {sor_residual:>14.3e}")
print()
print(f"max |phi_J - phi_SOR| = {phi_difference:.3e}")
print(f"max |E_J - E_SOR|     = {E_difference:.3e}")
print(f"TV(P_J, P_SOR)         = {tv_distance:.3e}")
print(f"speedup                 = {speedup:.2f}x")


solver           iterations      time, s       residual
Jacobi                 7060     0.605203      9.972e-06
Red-Black SOR           450     0.309913      9.295e-06

max |phi_J - phi_SOR| = 6.798e-03
max |E_J - E_SOR|     = 9.266e-04
TV(P_J, P_SOR)         = 7.455e-04
speedup                 = 1.95x


In [6]:
from dataclasses import replace
import pandas as pd

def make_field(cfg):
    field = ElectricField(cfg)
    field.channel[:30, cfg.nx // 2] = True
    field.apply_boundary_conditions()
    return field


omegas = np.arange(1.0, 1.96, 0.05)
results = []

for omega in omegas:
    cfg = replace(config, omega=round(float(omega), 2))

    times = []
    iterations = []
    residuals = []

    for _ in range(3):
        field = make_field(cfg)

        start = time.perf_counter()
        n_iter, residual = field.solve_laplace_sor()
        elapsed = time.perf_counter() - start

        times.append(elapsed)
        iterations.append(n_iter)
        residuals.append(residual)

    results.append({
        "omega": cfg.omega,
        "iterations": int(np.median(iterations)),
        "time": np.median(times),
        "residual": np.max(residuals),
    })

results = pd.DataFrame(results)

display(results)

best = results.loc[results["time"].idxmin()]

print(f"Best omega: {best['omega']:.2f}")
print(f"Iterations: {int(best['iterations'])}")
print(f"Median time: {best['time']:.6f} s")
print(f"Residual: {best['residual']:.3e}")

,omega,iterations,time,residual
0,1.00,4550,3.108826,0.000010
1,1.05,4180,2.518013,0.000010
2,1.10,3850,2.324357,0.000010
3,1.15,3540,2.188121,0.000010
4,1.20,3260,2.045381,0.000010
5,1.25,2990,1.885588,0.000010
6,1.30,2740,1.696045,0.000010
7,1.35,2510,1.562184,0.000010
8,1.40,2280,1.388320,0.000010
9,1.45,2070,1.265255,0.000010


Best omega: 1.95
Iterations: 250
Median time: 0.152985 s
Residual: 9.226e-06


In [7]:
omegas = [1.95, 1.96, 1.97, 1.98, 1.99]

results = []

for omega in omegas:
    cfg = replace(config, omega=omega)

    times = []
    iterations = []
    residuals = []

    for _ in range(5):
        field = make_field(cfg)

        start = time.perf_counter()
        n_iter, residual = field.solve_laplace_sor()
        times.append(time.perf_counter() - start)

        iterations.append(n_iter)
        residuals.append(residual)

    results.append({
        "omega": omega,
        "iterations": int(np.median(iterations)),
        "time": np.median(times),
        "residual": np.max(residuals),
    })

results = pd.DataFrame(results)
display(results)

best = results.loc[results["time"].idxmin()]
print(f"Best omega: {best['omega']:.2f}")

,omega,iterations,time,residual
0,1.95,250,0.256500,0.000009
1,1.96,210,0.173172,0.000008
2,1.97,250,0.208486,0.000009
3,1.98,350,0.292465,0.000010
4,1.99,660,0.418888,0.000008


Best omega: 1.96
